In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [11]:
# QASA NLI claim recall evaluation from subclaims file
import json
import re
import os
import numpy as np
from tqdm import tqdm
from transformers import pipeline
from google.colab import drive
drive.mount('/content/drive')

# -----------------------------
# CONFIG
# -----------------------------
SUBCLAIMS_FILE = "/content/drive/MyDrive/results2/qasa_subclaims/gold_subclaims_gpt54_mini_extracted.json"
RESULTS_DIR = "/content/drive/MyDrive/results2/qasa"
OUTPUT_FILE = "/content/drive/MyDrive/results2/citation_results/qasa_claim_recall_scores.json"
EXAMPLES_DIR = "/content/drive/MyDrive/results2/citation_results/claim_recall_examples"
os.makedirs(EXAMPLES_DIR, exist_ok=True)

# Thresholds for good/bad classification
GOOD_THRESHOLD = 0.75  # claim recall >= 75% → good example
BAD_THRESHOLD = 0.25   # claim recall <= 25% → bad example

# -----------------------------
# LOAD NLI MODEL
# -----------------------------
nli = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=0,
    truncation=True,
    max_length=1024,
)

# -----------------------------
# HELPERS
# -----------------------------
def remove_citations(text):
    return re.sub(r"\[\d+\]", "", text).strip()

def run_nli(premise, hypothesis, threshold=0.5):
    result = nli(f"{premise} {hypothesis}")[0]
    return 1 if (result["label"].lower() == "entailment" and result["score"] >= threshold) else 0

# -----------------------------
# LOAD SUBCLAIMS
# -----------------------------
with open(SUBCLAIMS_FILE) as f:
    subclaims_data = json.load(f)

subclaims_lookup = {}
for item in subclaims_data:
    question = item["question"]
    if "claims" in item:
        subclaims_lookup[question] = item["claims"]
    elif "subclaims_output" in item:
        raw = item["subclaims_output"]
        claims = re.findall(r"Subclaim \d+:\s*(.+?)(?=\nSubclaim \d+:|$)", raw, re.DOTALL)
        subclaims_lookup[question] = [c.strip() for c in claims if c.strip()]

# -----------------------------
# COMPUTE CLAIM RECALL
# -----------------------------
def compute_claims(data, subclaims_lookup):
    scores = []
    missing = 0
    good_examples = []
    bad_examples = []

    for item in tqdm(data):
        question = item.get("question", "")
        output = item.get("output", "")

        if not output or not output.strip():
            continue

        claims = subclaims_lookup.get(question)
        if not claims:
            missing += 1
            continue

        normalized_output = remove_citations(output)

        # Track per-claim entailment for detailed logging
        claim_results = [
            {"claim": claim, "entailed": bool(run_nli(normalized_output, claim))}
            for claim in claims
        ]
        entailed_count = sum(c["entailed"] for c in claim_results)
        item_recall = entailed_count / len(claims)
        scores.append(item_recall)

        example_record = {
            "question": question,
            "output": output,
            "claims": claim_results,
            "claim_recall": round(item_recall, 4),
            "entailed": entailed_count,
            "total_claims": len(claims),
        }

        if item_recall >= GOOD_THRESHOLD:
            good_examples.append(example_record)
        elif item_recall <= BAD_THRESHOLD:
            bad_examples.append(example_record)

    print(f"  Items missing from subclaims lookup: {missing}")
    print(f"  Good examples: {len(good_examples)} | Bad examples: {len(bad_examples)}")
    return 100 * np.mean(scores) if scores else 0.0, good_examples, bad_examples

# -----------------------------
# RUN OVER RESULTS FOLDER
# -----------------------------
results = {}

for filename in os.listdir(RESULTS_DIR):
    if not filename.endswith(".json"):
        continue

    path = os.path.join(RESULTS_DIR, filename)
    with open(path) as f:
        raw = json.load(f)
        data = raw["data"] if isinstance(raw, dict) else raw

    print(f"\nEvaluating {filename}...")
    score, good_examples, bad_examples = compute_claims(data, subclaims_lookup)
    results[filename] = {"claim_recall": round(score, 4)}
    print(f"  => Claim Recall: {score:.2f}%")

    # Save good/bad examples per file
    stem = os.path.splitext(filename)[0]
    examples_output = {
        "file": filename,
        "claim_recall": round(score, 4),
        "thresholds": {"good": GOOD_THRESHOLD, "bad": BAD_THRESHOLD},
        "good_examples": good_examples,
        "bad_examples": bad_examples,
    }
    examples_path = os.path.join(EXAMPLES_DIR, f"{stem}_examples.json")
    with open(examples_path, "w") as f:
        json.dump(examples_output, f, indent=2)
    print(f"  => Examples saved to {examples_path}")

with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nScores saved to {OUTPUT_FILE}")
print(json.dumps(results, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


Evaluating tot_qasa-gemma-4-26b-a4b-it-shot1-ndoc3-42-quick_test200.json...


100%|██████████| 200/200 [00:15<00:00, 13.19it/s]


  Items missing from subclaims lookup: 0
  Good examples: 47 | Bad examples: 103
  => Claim Recall: 35.92%
  => Examples saved to /content/drive/MyDrive/results2/citation_results/claim_recall_examples/tot_qasa-gemma-4-26b-a4b-it-shot1-ndoc3-42-quick_test200_examples.json

Evaluating cot_llama_quick200_rerun.formatted.json...


...

  Items missing from subclaims lookup: 0
  Good examples: 54 | Bad examples: 90
  => Claim Recall: 40.75%
  => Examples saved to /content/drive/MyDrive/results2/citation_results/claim_recall_examples/vanilla_qasa-gemma-4-26b-a4b-it-None-shot1-ndoc3-42-quick_test200_examples.json

Evaluating vanilla_qasa-llama-3.3-70b-instruct-None-shot1-ndoc3-42-quick_test200.json...


100%|██████████| 200/200 [00:06<00:00, 29.75it/s]


  Items missing from subclaims lookup: 0
  Good examples: 56 | Bad examples: 87
  => Claim Recall: 42.00%
  => Examples saved to /content/drive/MyDrive/results2/citation_results/claim_recall_examples/vanilla_qasa-llama-3.3-70b-instruct-None-shot1-ndoc3-42-quick_test200_examples.json

Evaluating qasa-gemma-4-26b-a4b-it-cot_1shot_second_demo_eval_gemma26-shot1-ndoc3-42-quick_test200.json...


100%|██████████| 200/200 [00:07<00:00, 28.03it/s]

  Items missing from subclaims lookup: 0
  Good examples: 72 | Bad examples: 70
  => Claim Recall: 50.50%
  => Examples saved to /content/drive/MyDrive/results2/citation_results/claim_recall_examples/qasa-gemma-4-26b-a4b-it-cot_1shot_second_demo_eval_gemma26-shot1-ndoc3-42-quick_test200_examples.json

Scores saved to /content/drive/MyDrive/results2/citation_results/qasa_claim_recall_scores.json
{
  "tot_qasa-gemma-4-26b-a4b-it-shot1-ndoc3-42-quick_test200.json": {
    "claim_recall": 35.9167
  },
  "cot_llama_quick200_rerun.formatted.json": {
    "claim_recall": 51.0833
  },
  "vanilla_qasa-gemma-4-26b-a4b-it-None-shot1-ndoc3-42-quick_test200.json": {
    "claim_recall": 40.75
  },
  "vanilla_qasa-llama-3.3-70b-instruct-None-shot1-ndoc3-42-quick_test200.json": {
    "claim_recall": 42.0
  },
  "qasa-gemma-4-26b-a4b-it-cot_1shot_second_demo_eval_gemma26-shot1-ndoc3-42-quick_test200.json": {
    "claim_recall": 50.5
  }
}


In [10]:
# eli5 compute subclaims
import json
import os
import re
import numpy as np
from tqdm import tqdm
from transformers import pipeline
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = "/content/drive/MyDrive/results2/eli5"
OUTPUT_FILE = "/content/drive/MyDrive/results2/citation_results/claim_recall_eli5.json"

nli = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=0,
    truncation=True,
    max_length=1024,
)

def remove_citations(text):
    return re.sub(r"\[\d+\]", "", text).strip()

def run_nli(premise, hypothesis, threshold=0.5):
    result = nli(f"{premise} {hypothesis}")[0]
    return 1 if (result["label"].lower() == "entailment" and result["score"] >= threshold) else 0

def compute_claims(data):
    scores = []
    skipped = 0

    for item in tqdm(data):
        output = item.get("output", "")
        claims = item.get("claims", [])

        if not output or not output.strip() or not claims:
            skipped += 1
            continue

        normalized_output = remove_citations(output)
        entail = sum(run_nli(normalized_output, claim) for claim in claims)
        scores.append(entail / len(claims))

    print(f"  Skipped (no output or no claims): {skipped}")
    return 100 * np.mean(scores) if scores else 0.0

results = {}

for filename in os.listdir(RESULTS_DIR):
    if not filename.endswith(".json"):
        continue

    path = os.path.join(RESULTS_DIR, filename)
    with open(path) as f:
        raw = json.load(f)
        data = raw["data"] if isinstance(raw, dict) else raw

    print(f"\nEvaluating {filename}...")
    score = compute_claims(data)
    results[filename] = {"claim_recall": round(score, 4)}
    print(f"  => Claim Recall: {score:.2f}%")

with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {OUTPUT_FILE}")
print(json.dumps(results, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


Evaluating tot_ eli5-gemma-4-26b-a4b-it-shot1-ndoc3-42-quick_test200.json...


100%|██████████| 200/200 [00:23<00:00,  8.50it/s]


  Skipped (no output or no claims): 0
  => Claim Recall: 18.83%

Evaluating cot_eli5-gemma-4-26b-a4b-it-gemma26-shot1-ndoc3-42-quick_test200.json...


100%|██████████| 200/200 [00:11<00:00, 17.92it/s]


  Skipped (no output or no claims): 0
  => Claim Recall: 9.50%

Evaluating vanilla_eli5-gemma-4-26b-a4b-it-openrouter-shot1-ndoc3-42-quick_test200.json...


100%|██████████| 200/200 [00:11<00:00, 18.05it/s]

  Skipped (no output or no claims): 0
  => Claim Recall: 8.17%

Saved to /content/drive/MyDrive/results2/citation_results/claim_recall_eli5.json
{
  "tot_ eli5-gemma-4-26b-a4b-it-shot1-ndoc3-42-quick_test200.json": {
    "claim_recall": 18.8333
  },
  "cot_eli5-gemma-4-26b-a4b-it-gemma26-shot1-ndoc3-42-quick_test200.json": {
    "claim_recall": 9.5
  },
  "vanilla_eli5-gemma-4-26b-a4b-it-openrouter-shot1-ndoc3-42-quick_test200.json": {
    "claim_recall": 8.1667
  }
}
